In [3]:
import os
import re
import glob
from bs4 import BeautifulSoup
import hashlib

In [18]:
shelves = {
    "Biographies": '/Users/SFL/notes/content/📖 Books 阅读笔记/🖋️ Biographies 传记',
    "Humanities": '/Users/SFL/notes/content/📖 Books 阅读笔记/🌍 Humanities 人文',
    "Personal Growth": '/Users/SFL/notes/content/📖 Books 阅读笔记/🌱 Personal Growth 个人成长',
    "Sciences": '/Users/SFL/notes/content/📖 Books 阅读笔记/🔬 Sciences 科学'
}

Deleting empty notes

In [223]:
def is_empty_or_generated_only(file_path):
    # return True
    """Checks if a file contains only auto-generated content (subtitles) and no actual written content."""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read().strip()
            content = re.sub(r"^---.*?---", "", content, flags=re.DOTALL).strip()
            subtitle_pattern = re.compile(r"^## — .+", re.MULTILINE)
            content_without_subtitles = subtitle_pattern.sub("", content).strip()

            return len(content_without_subtitles) == 0

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return False  # Default to not deleting if there's an error

def clear_folder(folder_path):
    """Deletes only notes that are empty or contain only generated subtitles."""
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename in exclude_files:
                continue
            
            file_path = os.path.join(folder_path, filename)

            try:
                if os.path.isfile(file_path) and filename.endswith(".md"):
                    if is_empty_or_generated_only(file_path):
                        os.remove(file_path)
                    else:
                        print(f"Kept: {os.path.splitext(filename)[0]}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
    else:
        print(f"Folder does not exist: {folder_path}")

for category, folder_path in shelves.items():
    clear_folder(folder_path)

print("Cleanup completed!")

Cleanup completed!


Creating book links

In [17]:
def update_shelf_notes(shelves):
    for shelf_name, folder_path in shelves.items():
        main_note_path = os.path.join(folder_path, "index.md")  # Ensure matching case

        if not os.path.isfile(main_note_path):
            
            # Create a new file named 'index.md' at the same directory
            index_note_path = os.path.join(os.path.dirname(main_note_path), "index.md")
            
            with open(index_note_path, "w", encoding="utf-8") as f:
                f.write(f"""---
title: {shelf_name.capitalize()}
---

""")  # Initial content
            print(f"Created new file: {index_note_path}")


        # Collect titles of all other .md files in the folder
        book_titles = [
            os.path.splitext(filename)[0]  # Remove .md extension
            for filename in os.listdir(folder_path)
            if filename.endswith(".md") and filename != "index.md"  # Exclude the main note
        ]

        # Format book titles as Obsidian links (sorted A-Z)
        book_links = "\n".join([f"[[{title}]]" for title in sorted(book_titles)])

        # Update the main note
        try:
            with open(main_note_path, "r", encoding="utf-8") as file:
                content = file.read()

            # Find the marker string and truncate everything after it
            marker = "**Full List of Books in this Category A-Z**"
            if marker in content:
                content = content.split(marker)[0] + marker + "\n" + book_links  # No extra newline
            else:
                content += f"\n\n{marker}\n" + book_links  # Ensure marker exists, no extra line

            # Write the updated content back to the main note
            with open(main_note_path, "w", encoding="utf-8") as file:
                file.write(content)

            print(f"Updated note: {main_note_path}")
        except Exception as e:
            print(f"Error updating {main_note_path}: {e}")

# Run the function
update_shelf_notes(shelves)

Updated note: /Users/SFL/notes/content/Books 阅读笔记/Other Genres 其他/Biographies 传记/index.md
Updated note: /Users/SFL/notes/content/Books 阅读笔记/Self-help 自我提升/Personal Development 成长/index.md
Updated note: /Users/SFL/notes/content/Books 阅读笔记/Self-help 自我提升/Business Thinking 商业/index.md
Updated note: /Users/SFL/notes/content/Books 阅读笔记/Other Genres 其他/Fiction 小说/index.md
Updated note: /Users/SFL/notes/content/Books 阅读笔记/Other Genres 其他/Humanities 人文/index.md
Updated note: /Users/SFL/notes/content/Books 阅读笔记/Natural & Social Sciences 科学/index.md


Importing highlights

In [1]:
import os
import re
import glob
from bs4 import BeautifulSoup
import hashlib

OBSIDIAN_DIR = '/Users/SFL/Library/Mobile Documents/iCloud~md~obsidian/Documents/content'

def generate_hash(text):
    return hashlib.md5(text.encode()).hexdigest()[:6]

def extract_book_title_and_quotes(html_path):
    with open(html_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    # Book title
    book_title_elem = soup.find("div", class_="bookTitle")
    book_title = book_title_elem.get_text(strip=True) if book_title_elem else None

    # All relevant DIVs
    elements = soup.find_all("div", class_=["sectionHeading", "noteHeading", "noteText"])
    
    content_lines = []
    pending_meta = None
    current_chapter = None

    for elem in elements:
        cls = elem.get("class", [])

        # Section heading → H3
        if "sectionHeading" in cls:
            sec = elem.get_text(strip=True)
            sec = re.sub(r'[\[\]"]', '', sec).title()
            content_lines.append(f"### {sec}")

        # noteHeading → decide highlight vs note
        elif "noteHeading" in cls:
            meta = elem.get_text(" ", strip=True)
            if "Highlight" in meta:
                # optional chapter sub-heading (H4)
                parts = meta.split(" - ", 1)
                if len(parts) > 1:
                    chap = parts[1].split(" > ")[0].strip()
                    if not chap.lower().startswith("page "):
                        chap = re.sub(r'[\[\]"]', '', chap).title()
                        if chap != current_chapter:
                            content_lines.append(f"#### {chap}")
                            current_chapter = chap
                pending_meta = "highlight"

            elif "Note" in meta:
                pending_meta = "note"

        # noteText → emit highlight or note
        elif "noteText" in cls and pending_meta:
            txt = elem.get_text(strip=True)

            if pending_meta == "highlight":
                # bare text
                content_lines.append(txt)

            elif pending_meta == "note":
                # single '> ' prefix
                note = f"> =={txt}=="
                # attach to last highlight (bare line)
                for i in range(len(content_lines)-1, -1, -1):
                    line = content_lines[i]
                    # skip headings and other notes
                    if not line.startswith("###") and not line.startswith("####") and not line.startswith(">"):
                        content_lines[i] = line + "\n" + note
                        break

            pending_meta = None

    return book_title, content_lines


def update_obsidian_note(note_path, highlights):
    preserved_hashes, highlight_extra_map = parse_existing_highlights(note_path)

    with open(note_path, "r", encoding="utf-8") as f:
        original = f.read()

    # reset Select Quotes block
    if "## Select Quotes" in original:
        original = re.sub(r"## Select Quotes\n.*", "## Select Quotes", original, flags=re.DOTALL).strip()
    else:
        original += "\n\n## Select Quotes"

    new_block = []
    for line in highlights:
        # H3 or H4
        if line.startswith("###"):
            new_block.append("\n" + line)
        # a note
        elif line.startswith("> "):
            new_block.append("\n" + line)
        # a bare highlight
        else:
            new_block.append("\n" + line)
            h = generate_hash(line)
            if h in preserved_hashes:
                new_block.append(highlight_extra_map[h])

    new_content = "\n".join(new_block) + "\n"

    with open(note_path, "w", encoding="utf-8") as f:
        f.write(original + "\n" + new_content)


def find_matching_note(book_title):
    """Finds the best-matching note based on the beginning of the HTML title."""
    b = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", book_title)
    note_files = glob.glob(f"{OBSIDIAN_DIR}/**/*.md", recursive=True)
    note_titles = {os.path.splitext(os.path.basename(f))[0]: f for f in note_files}

    for note_title in note_titles:
        n = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", note_title)
        if b.lower().startswith(n.lower()):
            return note_titles[note_title]

    return None

def parse_existing_highlights(note_path):
    """Parses existing highlights and creates hash mappings for notes & identifiers."""
    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    in_select_quotes = False
    preserved_hashes = set()
    highlight_extra_map = {}

    lines = content.splitlines()
    lines = [s for s in lines if s.strip()]
    
    for i, line in enumerate(lines):
        if "## Select Quotes" in line:
            in_select_quotes = True
            continue

        if in_select_quotes:
            if line.startswith("> ") and not line.startswith("> >"):
                hash_val = generate_hash(line)
                has_note = bool(i + 1 < len(lines) and lines[i + 1].strip().startswith("> >"))
                has_id = bool(i + 1 < len(lines) and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 1].strip()))
                has_both = bool(i + 2 < len(lines) and lines[i + 1].strip().startswith("> >") and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 2].strip()))

                if has_both:
                    preserved_hashes.add(hash_val)
                    highlight_extra_map[hash_val] = lines[i + 1] + '\n' + lines[i + 2]
                elif has_note or has_id:
                    preserved_hashes.add(hash_val)
                    highlight_extra_map[hash_val] = lines[i + 1]
                
    return preserved_hashes, highlight_extra_map
    """
    Updates the given Obsidian note with extracted highlights while preserving
    manually added notes and identifiers.
    """
    preserved_hashes, highlight_extra_map = parse_existing_highlights(note_path)

    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    if "## Select Quotes" in content:
        content = re.sub(r"## Select Quotes\n.*", "## Select Quotes", content, flags=re.DOTALL).strip()
    else:
        content += "\n\n## Select Quotes"

    new_content = []

    for i, line in enumerate(highlights):
        if line.strip().startswith("###"):
            new_content.append('\n' + line)
        elif line.strip().startswith("> ") and not line.startswith("> >"):
            new_content.append('\n' + line)
            highlight_hash = generate_hash(line)
            if highlight_hash in preserved_hashes:
                new_content.append(highlight_extra_map[highlight_hash]) 
        elif line.startswith("> >"):
            previous_hash = generate_hash(highlights[i - 1])
            if previous_hash in preserved_hashes and '> >' in highlight_extra_map[previous_hash]:
                continue
            else:
                new_content.append(line)

    new_content = "\n".join(new_content) + "\n"

    with open(note_path, "w", encoding="utf-8") as file:
        file.write(content + '\n' + new_content)

def process_kindle_highlights(html_path):
    """
    Extracts Kindle highlights from the provided HTML file and updates the
    corresponding Obsidian note.
    """
    book_title, highlights = extract_book_title_and_quotes(html_path)
    print(book_title)

    if not book_title:
        print(f"Could not extract book title from {html_path}")
        return

    note_path = find_matching_note(book_title)
    print(note_path)
    
    if not note_path:
        print(f"No matching Obsidian note found for: {book_title}")
        return

    update_obsidian_note(note_path, highlights)

In [2]:
HIGHLIGHTS_DIR = "/Users/SFL/Downloads"

def process_all_highlights():
    # Iterate over all files in the directory
    for file_name in os.listdir(HIGHLIGHTS_DIR):
        file_path = os.path.join(HIGHLIGHTS_DIR, file_name)
        
        # Process only files (skip directories or invalid files)
        if os.path.isfile(file_path) and file_name.endswith(".html"):
            process_kindle_highlights(file_path)

process_all_highlights()

Death by Black Hole And Other Cosmic Quandaries
/Users/SFL/Library/Mobile Documents/iCloud~md~obsidian/Documents/content/Books/Death by Black Hole.md
